# SAM31 segmentation on a Colab GPU

Upload `AI-assisted-DBH-measurement-pipeline` (codes/segmentation, model/SAM31, data/rawdata,
environment/python/sam31_gpu, and
base-point CSVs) to My Drive. This notebook uses the same shared implementation
as the local CPU entry point, including lossless individual candidate archives.
Select a GPU runtime. Outputs and selected masks go directly to
`output/SAM3SegResults`. To calculate DBH on Windows, synchronize the same project
folder from Drive; C++ reads the target masks there without a separate mask-copy step.
Local CPU execution and Windows C++ builds are documented in the repository README.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/AI-assisted-DBH-measurement-pipeline')
assert (PROJECT_ROOT / 'model/SAM31/checkpoints/sam3.1_multiplex.pt').is_file()


## Install supporting packages
Keep Colab's matching PyTorch/torchvision pair. If pip changes NumPy in an already
used session, restart the runtime before importing the model. This notebook's
GPU execution requires a live Colab GPU runtime.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r',
    str(PROJECT_ROOT / 'environment/python/sam31_gpu/requirements.txt')])


In [ ]:
import sys, importlib, torch
sys.path.insert(0, str(PROJECT_ROOT / 'codes/segmentation'))
import batchProcess_SAM31_CPU as segmentation
import selectTargetTree_SAM31 as selection
segmentation = importlib.reload(segmentation)
selection = importlib.reload(selection)
assert torch.cuda.is_available(), 'Choose a GPU runtime first.'
print(torch.__version__, torch.cuda.get_device_name(0))
print(segmentation.locateSam31SourceDirectory())
DATASET = 'all'              # 'natural', 'plotted', or 'all'
LIMIT_PER_DATASET = 1         # Start with 1; set 0 for the full dataset
GPU_PRECISION = 'float32'     # float16 uses less GPU memory


In [ ]:
images = []
for name, folder in [('natural', segmentation.NATURAL_FIELD_IMAGE_DIRECTORY),
                     ('plotted', segmentation.PLOTTED_FIELD_IMAGE_DIRECTORY)]:
    if DATASET in ('all', name):
        items = sorted(folder.glob('*.jpeg'))
        images.extend(items[:LIMIT_PER_DATASET] if LIMIT_PER_DATASET else items)
timings = segmentation.segmentImageBatchAndRecordTimings(
    images, device='cuda:0', precision=GPU_PRECISION, cpuThreadCount=2, warmupPassCount=1)
run_id = timings.stem.removeprefix('timings_')
selection.selectRun(run_id, DATASET)
print('C++ can read target masks from:', selection.SEG_RESULTS / 'targetTreeMasks')
